# Step 1 & 2: Data Pipeline
## Loading CodeSearchNet and Injecting Typos

## Background

**CodeSearchNet** is a dataset of ~500k Python functions scraped from GitHub.
Each function comes with a docstring — giving us clean `(code, comment)` pairs for free.

We inject typos artificially because:
- There is no large dataset of "comment with typo → correct comment" pairs
- We need to create our own training signal

**BERT** — A transformer model trained on millions of English sentences (Wikipedia, books).
It learns by playing a fill-in-the-blank game: `"The cat sat on the [MASK]"` → `"mat"`

**RoBERTa** — An improved version of BERT. General purpose language model.

**CodeBERT** — A bimodal transformer trained on both natural language (English) AND
programming languages (Python, Java, etc.) simultaneously. This is why we use it —
it understands the relationship between code and comments.

In [ ]:
import torch, json, random, re, pickle
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, RobertaForMaskedLM

In [ ]:
# Official CodeSearchNet — new organisation name as of 2025
# "python" selects only Python functions

dataset = load_dataset("code-search-net/code_search_net", "python", split="train")

print("Total examples:", len(dataset))
print("Fields:", dataset.column_names)

## CodeSearchNet Dataset Fields

| Field | Description |
|-------|-------------|
| `repository_name` | Which GitHub repo this came from |
| `func_path_in_repository` | File path inside that repo |
| `func_name` | Just the function name |
| `whole_func_string` | Full function — def + docstring + body |
| `func_code_string` | Just the body, no docstring |
| `func_code_tokens` | Body split into individual tokens |
| `func_documentation_string` | Just the docstring text (no quotes) — **our comment** |
| `func_documentation_tokens` | Docstring split into individual words |
| `split_name` | train / test / validation label |

CodeSearchNet scraped millions of open source Python files from GitHub.
For each function, a parser automatically separated the docstring from the code body
and stored them in separate fields.

We use:
- `whole_func_string` → fed to the model as **code context**
- `func_documentation_string` → the **comment we corrupt and ask the model to fix**

In [ ]:
def char_swap(word):
    if len(word)<3:
        return word
    i = random.randint(1, len(word)-2) #picj a position not first not last
    chars = list(word)
    chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def char_delete(word):
    if len(word)<3:
        return word
    i = random.randint(1, len(word)-2) # pick a position to delete
    return word[:i]+word[i+1:]

def char_insert(word):
    if len(word)<2:
        return word
    i = random.randint(1, len(word)-1)
    ch = random.choice("abcdefghijklmnopqrstuvwxyz")
    return word[:i]+ch+word[i:]

def char_substitute(word):
    # Map each key to its neighbors on a QWERTY keyboard
    neighbors = {
        'a':'sqwz', 'b':'vghn', 'c':'xdfv', 'd':'erfcs', 'e':'wrsdf',
        'f':'rtgvcd', 'g':'tyhbvf', 'h':'yuibnmg', 'i':'uojkl', 'j':'uikhbn',
        'k':'ioljmn', 'l':'opkm', 'm':'njkl', 'n':'bhjm', 'o':'iklp',
        'p':'ol', 'q':'wa', 'r':'etdf', 's':'awedxz', 't':'ryfg',
        'u':'yhij', 'v':'cfgb', 'w':'qase', 'x':'zsdc', 'y':'tughi', 'z':'asx'
    }
    if len(word) < 2:
        return word
    # Only pick positions where the character has known neighbors
    valid = [i for i, c in enumerate(word) if c.lower() in neighbors]
    if not valid:
        return word
    i = random.choice(valid)
    replacement = random.choice(neighbors[word[i].lower()])
    # Preserve original capitalisation
    if word[i].isupper():
        replacement = replacement.upper()
    return word[:i] + replacement + word[i+1:]

In [ ]:
STRATEGIES = [char_swap, char_delete, char_insert, char_substitute]

def corrupt_word(word):
    strategy = random.choice(STRATEGIES)
    result = strategy(word)
    if result == word:
        result = char_delete(word)
    return result

In [ ]:
# Test each strategy individually so we can see exactly what each one does
# We use words taken from the actual docstring we just saw

test_words = ["estimate", "discontinuity", "resolution", "segmentation", "returns"]

print("=" * 55)
print("TESTING EACH STRATEGY INDIVIDUALLY")
print("=" * 55)

for word in test_words:
    print(f"\nOriginal : {word}")
    print(f"  swap       → {char_swap(word)}")
    print(f"  delete     → {char_delete(word)}")
    print(f"  insert     → {char_insert(word)}")
    print(f"  substitute → {char_substitute(word)}")

print("\n" + "=" * 55)
print("TESTING corrupt_word (random strategy each time)")
print("=" * 55)

# Run corrupt_word 3 times on the same word to show randomness
for word in test_words:
    results = [corrupt_word(word) for _ in range(3)]
    print(f"{word:20s} → {results}")

In [ ]:
# corruption_rate = fraction of eligible words to corrupt
#   e.g. 0.15 means ~15% of words get a typo

# min_word_len = only corrupt words this long or longer

def inject_typos(comment, corruption_rate=0.15, min_word_len=4):
    
    tokens = comment.split()       # split comment into list of words
    if not tokens:
        return comment, []
    
    # Find which word positions are eligible for corruption
    # Eligible = long enough + only letters (skip things like ":return:")
    eligible = [
        i for i, word in enumerate(tokens)
        if len(word) >= min_word_len and re.fullmatch(r"[A-Za-z]+[.,!?]?", word)
    ]
    
    if not eligible:
        return comment, []         # nothing to corrupt, return as-is
    
    # How many words to corrupt?
    # At least 1, at most 3, based on corruption_rate
    n_corrupt = min(
        max(1, round(len(eligible) * corruption_rate)),
        3
    )
    
    # Randomly pick which positions to corrupt
    chosen = random.sample(eligible, min(n_corrupt, len(eligible)))
    
    # Keep a log of every change we make
    # We need this later so the model knows what the correct answer was
    corruption_log = []
    
    for idx in chosen:
        original_word = tokens[idx]
        
        # Handle punctuation attached to word e.g. "returns,"
        punct = ""
        if original_word[-1] in ".,!?":
            punct = original_word[-1]
            original_word = original_word[:-1]   # remove punct before corrupting
        
        corrupted_word = corrupt_word(original_word)
        tokens[idx] = corrupted_word + punct     # put punct back
        
        corruption_log.append({
            "word_idx"  : idx,
            "original"  : original_word + punct,
            "corrupted" : corrupted_word + punct,
        })
    
    corrupted_comment = " ".join(tokens)
    return corrupted_comment, corruption_log

In [ ]:
import json
processed = []
with open(r"C:\Users\thiru\autocorrect_comments\data\train_data.jsonl") as f:
    for line in f:
        processed.append(json.loads(line))
print(f"Loaded {len(processed):,} examples")

## Step 3: Tokenization

The model does not understand English or code — it only understands numbers.
**Tokenization** is the process of converting text into numbers.

**Subwords** — The tokenizer does not split word by word. It uses subwords:
- `"folder"` → single token (known word)
- `"foleder"` → split into parts (typo, unknown word)

**Input format to the model:**

    <s> code tokens here </s> comment with <mask> here </s>

- `<s>` — start of sequence, always first
- `</s>` — separator between code and comment
- `<mask>` — corrupted word replaced with blank for model to fill

**LSTM vs Transformer:**

| | LSTM | Transformer |
|--|------|-------------|
| Reads | Word by word | All words at once |
| Context | Forgets distant words | Attends to everything |
| Used in | SETU project | This project |

In [ ]:
#just loading the dictionary
from transformers import AutoTokenizer

# Load from local folder — no internet needed
tokenizer = AutoTokenizer.from_pretrained(
    r"C:\Users\thiru\autocorrect_comments\tokenizer"
)

print("Vocabulary size:", tokenizer.vocab_size)
print("Special tokens:", tokenizer.all_special_tokens)

In [ ]:
#we are taking one (code, corrupted_comment) pair from our saved data and converting it into numbers. We also replace the correpted word with <mask> so the model knows where to predict

# Take one example from our saved data
sample = processed[0]

print("Code (first 2 lines):")
print("\n".join(sample["code"].splitlines()[:2]))
print("\nOriginal comment :", sample["original"])
print("Corrupted comment:", sample["corrupted"])
print("What was changed :", sample["log"])

In [ ]:
#if we subword the corrupted word then the problem is that if we feed it like that, the model sees a broken word and has to gues what it means
#that is too hard . the model doesnt know where the mistake is or what kind of fix is needed

#when we use <mask> we are telling the model that something si missing at this exact position.Use the code context and the surrounding words to figure out what belongs here

corrupted_comment = sample["corrupted"]
log = sample["log"]

# Get the corrupted word from the log
corrupted_word = log[0]["corrupted"]

# Replace it with <mask> token
# Only replace the first occurrence to be safe
masked_comment = corrupted_comment.replace(corrupted_word, "<mask>", 1)

print("Corrupted comment:", corrupted_comment)
print("Masked comment   :", masked_comment)
print("Model must predict:", log[0]["original"])

In [ ]:
#we now feed the model code(context), masked comment combined
#format: <s> code </s> masked_comment</s>
#max_length  = 512 as RoBERTa can only handle 512 tokens at once
# truncation = true = if code is very long, cut it to fit
#return_tensors = "pt" = return pytorch tensors, not plain lists because the model expects pytorch format

code = sample["code"]

tokenized = tokenizer(
    code,                    # first sequence (code)
    masked_comment,          # second sequence (masked comment)
    max_length=512,
    truncation=True,
    padding="max_length",    # pad shorter inputs to exactly 512
    return_tensors="pt"
)

print("Keys in tokenized output:", list(tokenized.keys()))
print("\nShape of input_ids:", tokenized["input_ids"].shape)
print("\nFirst 20 token IDs:", tokenized["input_ids"][0][:20].tolist())
print("\nDecoded back to text:")
print(tokenizer.decode(tokenized["input_ids"][0][:30]))

In [ ]:
# input_ids = the actual token numbers. this is what the model reads
#attention_mask = tells the model which tokens are real and which are just padding
#1=real token(pay attention to this), 0=padding(ignore)

mask_token_id = tokenizer.mask_token_id
print("ID of <mask> token:", mask_token_id)

# Find its position in our input
input_ids_list = tokenized["input_ids"][0].tolist()
mask_position = input_ids_list.index(mask_token_id)
print("Position of <mask> in sequence:", mask_position)

# Decode tokens around the mask to verify it's in the right place
print("\nTokens around <mask> (positions -3 to +3):")
surrounding = tokenized["input_ids"][0][mask_position-3 : mask_position+4]
print(tokenizer.decode(surrounding))

In [ ]:
#cell 16
# Tokenize all 20,000 examples and store them
# This will take 5-10 minutes on CPU — normal, don't worry

import torch

all_input_ids = []
all_attention_masks = []
all_mask_positions = []
all_labels = []       # the correct word at each mask position

skipped = 0
mask_token_id = tokenizer.mask_token_id

for example in processed:
    
    # Step 1: replace corrupted word with <mask>
    corrupted_word = example["log"][0]["corrupted"]
    masked_comment = example["corrupted"].replace(corrupted_word, "<mask>", 1)
    
    # Step 2: tokenize code + masked comment together
    tokenized = tokenizer(
        example["code"],
        masked_comment,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    
    # Step 3: find position of <mask>
    input_ids_list = tokenized["input_ids"][0].tolist()
    if mask_token_id not in input_ids_list:
        # mask got truncated (code was too long) — skip this example
        skipped += 1
        continue
    mask_position = input_ids_list.index(mask_token_id)
    
    # Step 4: tokenize the correct word to get its ID (our label)
    # This is the answer the model should predict at mask_position
    correct_word = example["log"][0]["original"]
    correct_tokens = tokenizer.encode(correct_word, add_special_tokens=False)
    if len(correct_tokens) != 1:
        # Word splits into multiple subwords — skip for now, too complex
        skipped += 1
        continue
    correct_token_id = correct_tokens[0]
    
    all_input_ids.append(tokenized["input_ids"][0])
    all_attention_masks.append(tokenized["attention_mask"][0])
    all_mask_positions.append(mask_position)
    all_labels.append(correct_token_id)

print(f"Tokenized : {len(all_input_ids):,} examples")
print(f"Skipped   : {skipped:,} examples")
print(f"\nSample label: '{tokenizer.decode([all_labels[0]])}' (ID: {all_labels[0]})")
print(f"Sample mask position: {all_mask_positions[0]}")

In [ ]:
#Pickles handles python objects like tensors natively whereas JSONL only handles text/numbers
import pickle

save_path = r"C:\Users\thiru\autocorrect_comments\data\tokenized_data.pkl"

data_to_save = {
    "input_ids"       : torch.stack(all_input_ids),
    "attention_masks" : torch.stack(all_attention_masks),
    "mask_positions"  : all_mask_positions,
    "labels"          : all_labels,
}

with open(save_path, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"Saved tokenized data → {save_path}")
print(f"Total examples saved: {len(all_labels):,}")
print(f"input_ids shape: {data_to_save['input_ids'].shape}")